# Children, Regions, and Timestamps

Timelines nest inside one another. A **timestamp** is a cross-section
that shows where you are in every active child at a given root coordinate.

In [1]:
from timetoalign import IdCoordinate, TimeUnit
from timetoalign.loader.score.partitura import PartituraLoader
from timetoalign.testdata import ensure_data
from timetoalign.timelines import ContinuousLogicalTimeline

/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Load a Structured Score

In [2]:
DATA_DIR = ensure_data("vienna_1x22")

loader = PartituraLoader()
loader.load(DATA_DIR / "Chopin_op10_no3.musicxml")
tl = loader.create_timeline(uid="chopin_etude")
tl

ContinuousLogicalTimeline(id='chopin_etude', length=83/2, unit=quarters, events=0, children=4, cmaps=3)

## Measures as Regions

The loader's `EventStore` contains measure intervals. We can filter
notes by coordinate bounds.

In [3]:
measures = loader.store.measures.to_dataframe()
m2 = measures.iloc[1]
{
    "measure": 2,
    "start": float(m2.start),
    "end": float(m2.end),
}

{'measure': 2, 'start': 0.5, 'end': 2.5}

In [4]:
tl.get_child("notes").get_events(
    event_type="Note", min_coord=float(m2.start), max_coord=float(m2.end)
).to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,mc,mn,mc_onset,mn_onset,specific_pitch,midi,tpc,octave,tied,gracenote,chord_id,voice,staff,part_id
0,notes:note:000002,E4,interval,Note,1/2,1,1/2,2,2,0,0,"{'step': 'E', 'alter': 0, 'octave': 4, 'cents'...",64,4,4,0,None,NaN,1,1,P1
1,notes:note:000003,G♯3,interval,Note,1/2,3/4,1/4,2,2,0,0,"{'step': 'G', 'alter': 1, 'octave': 3, 'cents'...",56,8,3,0,None,NaN,3,1,P1
2,notes:note:000004,E2,interval,Note,1/2,3/4,1/4,2,2,0,0,"{'step': 'E', 'alter': 0, 'octave': 2, 'cents'...",40,4,2,0,None,NaN,4,2,P1
3,notes:note:000005,E2,interval,Note,1/2,3/2,1,2,2,0,0,"{'step': 'E', 'alter': 0, 'octave': 2, 'cents'...",40,4,2,0,None,NaN,7,2,P1
4,notes:note:000006,B3,interval,Note,3/4,1,1/4,2,2,1/4,1/4,"{'step': 'B', 'alter': 0, 'octave': 3, 'cents'...",59,5,3,0,None,NaN,3,1,P1


## Create Children from Boundaries

A boundary list with *k*+1 coordinates creates *k* named children. The
children use their parent's concrete timeline class and tile each interval
without copying the parent's events.

In [5]:
movement = ContinuousLogicalTimeline(length=12, uid="movement")
phrases = movement.create_children_from_boundaries(
    [0, 4, 9, 12],
    names=["opening", "middle", "closing"],
)

[
    {
        "name": phrase.name,
        "offset": movement.get_child_offset(phrase.id),
        "length": phrase.length,
    }
    for phrase in phrases
]

[{'name': 'opening', 'offset': Coordinate(0, quarters), 'length': Coordinate(4, quarters)}, {'name': 'middle', 'offset': Coordinate(4, quarters), 'length': Coordinate(5, quarters)}, {'name': 'closing', 'offset': Coordinate(9, quarters), 'length': Coordinate(3, quarters)}]

Because the children cover the parent contiguously from 0 to 12, this
hierarchy has the structure of a segment line.

In [6]:
movement.is_segment_line()

True

## Resolve a Grandchild Coordinate

Nest the movement at offset 3 in a larger score. `get_coordinate()` follows
the entire descendant path: local coordinate 2 in the `middle` phrase first
gains that phrase's offset 4, then the movement's offset 3.

In [7]:
piece = ContinuousLogicalTimeline(length=20, uid="piece")
piece.add_child(movement, offset=3)

middle_coordinate = IdCoordinate(2, TimeUnit.quarters, "middle")
{
    "from IdCoordinate": piece.get_coordinate(middle_coordinate),
    "from value and timeline_id": piece.get_coordinate(2, timeline_id="middle"),
}

{'from IdCoordinate': Coordinate(Fraction(9, 1), quarters), 'from value and timeline_id': Coordinate(Fraction(9, 1), quarters)}

## Control Diagram Depth

The default recurses through the full hierarchy. `depth=1` keeps only the
root's direct children, which is useful for a compact overview.

In [8]:
print("Full hierarchy:")
print(piece.diagram(depth=True))
print("\nOne child level:")
print(piece.diagram(depth=1))

Full hierarchy:
ContinuousLogicalTimeline[piece] (1 children)
                      0 __________________________________ 20 quarters
  └─ movement         3      ____________________          15
     ├─ opening          3      ______                        7
     ├─ middle           7            _________               12
     └─ closing         12                     _____          15

One child level:
ContinuousLogicalTimeline[piece] (1 children)
                      0 __________________________________ 20 quarters
  └─ movement         3      ____________________          15


## Timestamps: Cross-Section Queries

At root coordinate X, what is the local coordinate inside each active
child timeline?

In [9]:
timestamps_df = tl.to_dataframe()
timestamps_df.head(10)

,axis (quarters),chopin_etude (quarters),notes (quarters),measures (quarters),controls (quarters),annotations (quarters),quarters_to_ticks (ticks),quarters_to_measures (floating_measures),raw_quarters (quarters)
id,,,,,,,,,
notes:note:000001,0,0,0,0,0,0,0,1,-1/2
notes:note:000002,1/2,1/2,1/2,1/2,1/2,1/2,240,2,0
notes:note:000006,3/4,3/4,3/4,3/4,3/4,3/4,360,17/8,1/4
notes:note:000008,1,1,1,1,1,1,480,9/4,1/2
notes:note:000010,5/4,5/4,5/4,5/4,5/4,5/4,600,19/8,3/4
notes:note:000013,3/2,3/2,3/2,3/2,3/2,3/2,720,5/2,1
notes:note:000018,7/4,7/4,7/4,7/4,7/4,7/4,840,21/8,5/4
notes:note:000020,2,2,2,2,2,2,960,11/4,3/2
notes:note:000021,9/4,9/4,9/4,9/4,9/4,9/4,1080,23/8,7/4


`NaN` means the coordinate falls outside that child's extent.

**Next:** [Timeline Groups](tut02_timeline_groups.ipynb)